In [5]:
cd /Users/karolinegriesbach/Documents/Innkeepr/Git/evaluation-and-execution-scripts

In [6]:
import os 
import time
import logging
import delta_sharing
import polars as pl

import general_functions.databricks_client as db_client
from general_functions.return_workspace_ids import return_workspace_ids
# logging.basicConfig(level=logging.DEBUG)

In [13]:
customer = "More"
workspace_id = return_workspace_ids()
workspace_id = [acc["id"] for acc in workspace_id if acc["name"] == customer]
workspace_id = workspace_id[0]
print(f"Workspace ID for {customer}: {workspace_id}")
storage_location = f"s3://{workspace_id}-databricks/__unitystorage/schemas/9cac5994-5881-43d4-981b-90e387cfa765/tables/abf7628b-cd21-4530-a0e0-65c5bcf0def8/"
storage_options = {
    "AWS_REGION": "eu-central-1",
    "AWS_ACCESS_KEY_ID": os.environ["AWS_ACCESS_KEY_ID"],
    "AWS_SECRET_ACCESS_KEY": os.environ["AWS_SECRET_ACCESS_KEY"],
}

In [8]:
start_time_pands = time.time()
profile_path = db_client.return_databricks_client()
table_path = f"{profile_path}#delta_share_events.{workspace_id}.features_view_30_outlook" 
df = delta_sharing.load_as_pandas(table_path)
total_time_pands = time.time() - start_time_pands
print(f"Time taken to load data into pandas: {total_time_pands:.2f} seconds")

# Read via delta table

https://delta-io.github.io/delta-rs/integrations/delta-lake-polars/#reading-a-delta-lake-table-with-polars

In [9]:
start_time_polars2 = time.time()
print("Loading data into polars...")
df_polars2 = pl.read_delta(storage_location, storage_options=storage_options)
#df_polars2 = pl.scan_delta(storage_location2).collect()  
total_time_polars2 = time.time() - start_time_polars2
print(f"Time taken to load data into polars: {total_time_polars2:.2f} seconds")
time_difference2 = total_time_polars2/total_time_pands
print(f"Time difference between pandas and polars: {time_difference2:.2f}")

In [10]:
if df_polars2.shape[0] != df.shape[0]:
    print(f"Warning: Number of rows in pandas ({df.shape[0]}) and polars ({df_polars2.shape[0]}) dataframes do not match.")

In [11]:
start_time_polars3 = time.time()
print("Loading data into polars...")
df_polars3 = pl.scan_delta(storage_location, storage_options=storage_options).collect()  
total_time_polars3 = time.time() - start_time_polars3
print(f"Time taken to load data into polars: {total_time_polars3:.2f} seconds")
time_difference3 = total_time_polars3/total_time_pands
print(f"Time difference between pandas and polars: {time_difference3:.2f}")

In [12]:
if df_polars3.shape[0] != df.shape[0]:
    print(f"Warning: Number of rows in pandas ({df.shape[0]}) and polars ({df_polars3.shape[0]}) dataframes do not match.")